# Module 09: Full CPA Attack Walkthrough — Lab

This lab performs a complete CPA attack, analyzes results, and computes PGE.

**Objectives:**
1. Configure and run a full CPA attack on 16-byte AES key
2. Visualize correlation heatmaps for all key bytes
3. Analyze key ranking and identify correct key
4. Compute PGE across multiple attack runs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 60)
print("FULL CPA ATTACK: 128-BIT AES KEY RECOVERY")
print("=" * 60)

# AES S-box
AES_SBOX = np.array([
    0x63,0x7C,0x77,0x7B,0xF2,0x6B,0x6F,0xC5,0x30,0x01,0x67,0x2B,0xFE,0xD7,0xAB,0x76,
    0xCA,0x82,0xC9,0x7D,0xFA,0x59,0x47,0xF0,0xAD,0xD4,0xA2,0xAF,0x9C,0xA4,0x72,0xC0,
    0xB7,0xFD,0x93,0x26,0x36,0x3F,0xF7,0xCC,0x34,0xA5,0xE5,0xF1,0x71,0xD8,0x31,0x15,
    0x04,0xC7,0x23,0xC3,0x18,0x96,0x05,0x9A,0x07,0x12,0x80,0xE2,0xEB,0x27,0xB2,0x75,
    0x09,0x83,0x2C,0x1A,0x1B,0x6E,0x5A,0xA0,0x52,0x3B,0xD6,0xB3,0x29,0xE3,0x2F,0x84,
    0x53,0xD1,0x00,0xED,0x20,0xFC,0xB1,0x5B,0x6A,0xCB,0xBE,0x39,0x4A,0x4C,0x58,0xCF,
    0xD0,0xEF,0xAA,0xFB,0x43,0x4D,0x33,0x85,0x45,0xF9,0x02,0x7F,0x50,0x3C,0x9F,0xA8,
    0x51,0xA3,0x40,0x8F,0x92,0x9D,0x38,0xF5,0xBC,0xB6,0xDA,0x21,0x10,0xFF,0xF3,0xD2,
    0xCD,0x0C,0x13,0xEC,0x5F,0x97,0x44,0x17,0xC4,0xA7,0x7E,0x3D,0x64,0x5D,0x19,0x73,
    0x60,0x81,0x4F,0xDC,0x22,0x2A,0x90,0x88,0x46,0xEE,0xB8,0x14,0xDE,0x5E,0x0B,0xDB,
    0xE0,0x32,0x3A,0x0A,0x49,0x06,0x24,0x5C,0xC2,0xD3,0xAC,0x62,0x91,0x95,0xE4,0x79,
    0xE7,0xC8,0x37,0x6D,0x8D,0xD5,0x4E,0xA9,0x6C,0x56,0xF4,0xEA,0x65,0x7A,0xAE,0x08,
    0xBA,0x78,0x25,0x2E,0x1C,0xA6,0xB4,0xC6,0xE8,0xDD,0x74,0x1F,0x4B,0xBD,0x8B,0x8A,
    0x70,0x3E,0xB5,0x66,0x48,0x03,0xF6,0x0E,0x61,0x35,0x57,0xB9,0x86,0xC1,0x1D,0x9E,
    0xE1,0xF8,0x98,0x11,0x69,0xD9,0x8E,0x94,0x9B,0x1E,0x87,0xE9,0xCE,0x55,0x28,0xDF,
    0x8C,0xA1,0x89,0x0D,0xBF,0xE6,0x42,0x68,0x41,0x99,0x2D,0x0F,0xB0,0x54,0xBB,0x16,
], dtype=np.uint8)

hw = np.vectorize(lambda b: bin(b).count('1'))

def pearson(x, y):
    n = len(x)
    mx, my = np.mean(x), np.mean(y)
    dx, dy = x - mx, y - my
    num = np.sum(dx * dy)
    den = np.sqrt(np.sum(dx**2) * np.sum(dy**2))
    return num / den if den > 0 else 0

# Setup
np.random.seed(42)
full_key = np.array([
    0x2B, 0x7E, 0x15, 0x16, 0x28, 0xAE, 0xD2, 0xA6,
    0xAB, 0xF7, 0x15, 0x88, 0x09, 0xCF, 0x4F, 0x3C
], dtype=np.uint8)

n_traces = 1000
n_samples = 250
plaintexts = np.random.randint(0, 256, (n_traces, 16), dtype=np.uint8)

# Generate traces with leakage from all 16 bytes
traces = np.random.normal(0, 0.3, (n_traces, n_samples))
for byte_idx in range(16):
    t_pos = 15 + byte_idx * 14  # Each byte at different time
    for i in range(n_traces):
        sbox_out = AES_SBOX[plaintexts[i, byte_idx] ^ full_key[byte_idx]]
        traces[i, t_pos] += hw(sbox_out) * 0.5

print(f"Traces: {n_traces}, Samples: {n_samples}")
print(f"Key to recover: {[f'0x{b:02X}' for b in full_key]}")

In [ ]:
# Run CPA attack on all 16 bytes
print("\n" + "=" * 60)
print("CPA ATTACK RESULTS")
print("=" * 60)

recovered_key = np.zeros(16, dtype=np.uint8)
all_correlations = []
all_rankings = []

for byte_idx in range(16):
    max_corrs = np.zeros(256)
    
    for k_guess in range(256):
        intermediates = hw(AES_SBOX[plaintexts[:, byte_idx] ^ k_guess]).astype(float)
        t_pos = 15 + byte_idx * 14
        max_corrs[k_guess] = abs(pearson(intermediates, traces[:, t_pos]))
    
    # Rank key guesses
    ranking = np.argsort(-max_corrs)
    all_rankings.append(ranking)
    all_correlations.append(max_corrs)
    
    recovered_key[byte_idx] = ranking[0]
    correct_rank = np.where(ranking == full_key[byte_idx])[0][0] + 1
    status = '✓' if ranking[0] == full_key[byte_idx] else '✗'
    
    print(f"Byte {byte_idx:2d}: True=0x{full_key[byte_idx]:02X}  "
          f"Recovered=0x{ranking[0]:02X}  Rank={correct_rank:3d}  "
          f"Max ρ={max_corrs[ranking[0]]:.4f}  {status}")

print(f"\nFull key match: {'YES' if np.array_equal(recovered_key, full_key) else 'NO'}")
print(f"Recovered: {[f'0x{b:02X}' for b in recovered_key]}")
print(f"Original:  {[f'0x{b:02X}' for b in full_key]}")

In [ ]:
# Correlation Heatmap for all 16 bytes
print("\n" + "=" * 60)
print("CORRELATION HEATMAPS")
print("=" * 60)

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for byte_idx in range(16):
    ax = axes[byte_idx]
    
    # Compute full correlation matrix for this byte
    corr_matrix = np.zeros((n_samples, 256))
    for k_guess in range(256):
        intermediates = hw(AES_SBOX[plaintexts[:, byte_idx] ^ k_guess]).astype(float)
        for t in range(n_samples):
            corr_matrix[t, k_guess] = abs(pearson(intermediates, traces[:, t]))
    
    # Plot heatmap
    im = ax.imshow(corr_matrix.T, aspect='auto', cmap='hot', origin='lower',
                   vmin=0, vmax=0.8)
    ax.axvline(x=15+byte_idx*14, color='cyan', linestyle='--', alpha=0.5)
    ax.axhline(y=full_key[byte_idx], color='lime', linestyle='--', alpha=0.5)
    ax.set_title(f'Byte {byte_idx} (key=0x{full_key[byte_idx]:02X})', fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('CPA Correlation Heatmaps for All 16 Key Bytes', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# PGE (Probability of Guessing Entropy) calculation
print("=" * 60)
print("PGE CALCULATION")
print("=" * 60)

def run_cpa_single_byte(traces_sub, pt_sub, byte_idx, true_key_byte):
    """Run CPA on a subset of traces for one byte, return rank of correct key"""
    max_corrs = np.zeros(256)
    for k_guess in range(256):
        intermediates = hw(AES_SBOX[pt_sub[:, byte_idx] ^ k_guess]).astype(float)
        t_pos = 15 + byte_idx * 14
        max_corrs[k_guess] = abs(pearson(intermediates, traces_sub[:, t_pos]))
    ranking = np.argsort(-max_corrs)
    return np.where(ranking == true_key_byte)[0][0] + 1  # rank (1-indexed)

# Compute PGE for different trace counts
trace_counts = [100, 200, 500, 1000]
n_runs = 50

print(f"Running {n_runs} attack iterations per trace count...")
print(f"{'Traces':>8} {'PGE (byte 0)':>15} {'Avg Rank':>12}")
print("-" * 40)

for n_sub in trace_counts:
    pge_count = 0
    ranks = []
    for run in range(n_runs):
        idx = np.random.choice(n_traces, min(n_sub, n_traces), replace=False)
        rank = run_cpa_single_byte(traces[idx], plaintexts[idx], 0, full_key[0])
        ranks.append(rank)
        if rank == 1:
            pge_count += 1
    pge = pge_count / n_runs
    avg_rank = np.mean(ranks)
    print(f"{n_sub:>8} {pge:>15.4f} {avg_rank:>12.2f}")

print("\nPGE = P(correct key has rank 1)")
print("PGE of 1.0 means the attack always succeeds with that trace count.")